In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/MyDrive

In [ ]:
import os

repo_path = '/content/drive/MyDrive/bitenet'
if os.path.isdir(repo_path):
    %cd /content/drive/MyDrive/bitenet
    !git pull origin fix/colab_notebook
else:
    !git clone https://github.com/agataben/bitenet.git
    %cd bitenet
    !git checkout fix/colab_notebook

In [ ]:
%pip install -r "colab_requirements.txt"

In [ ]:
from src.utils import set_seed

seed = 1238
set_seed(seed)

In [ ]:
from src.utils import get_norm_parameters

yaml_path = 'data'
mean, std = get_norm_parameters(yaml_path = yaml_path)
print(mean)
print(std)

In [ ]:
from torchvision import transforms

train_transf = transforms.Compose([ transforms.Resize(256),
                                    transforms.CenterCrop(224),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean,std)
                                  ])

val_transf = transforms.Compose([ transforms.Resize(256),
                                  transforms.CenterCrop(224),
                                  transforms.ToTensor(),
                                  transforms.Normalize(mean,std)
                                ])


In [ ]:
from src.food101_dataset import Food101DataSet

data_root = 'data'
train_ds = Food101DataSet(data_root = data_root, csv = 'data/train.csv', transform = train_transf)
val_ds = Food101DataSet(data_root = data_root, csv = 'data/val.csv', transform = val_transf)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size = 1024, num_workers = 2, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = 1024, num_workers = 2, shuffle = False)
loaders = {'train': train_loader,
         'test': val_loader}

In [ ]:
from src.bitenet_v1 import BiteNetV1

model = BiteNetV1()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/bitenet/results/bitenet_v1/logs/exp_1"

In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
from src.training import train

exp_name = 'exp_1'
ckpt_dir = '/content/drive/MyDrive/bitenet/results/bitenet_v1/ckpt'
logdir = '/content/drive/MyDrive/bitenet/results/bitenet_v1/logs'
model = train(model, loaders, lr = 0.01, momentum = 0.99, epochs = 30,
              exp_name = exp_name, logdir = logdir, ckpt_dir = ckpt_dir)